# 📝 통계 기초 과제 LV3(통합) — 고객 데이터 기술통계·추론 리포트

> 하나의 고객 데이터를 **불러오기 → 정제 → 대표값·산포 → 분포·상관 → 신뢰구간 → 인사이트** 의 순서로 처음부터 끝까지 분석하는 통합 문제입니다. 각 `### N단계` 셀에 그 단계에서 할 일이 자립적으로 적혀 있어요.

## 풀이 방법
1. 문제마다 **1단계에서 데이터를 한 번 불러와** 같은 `df` 로 끝까지 이어 분석합니다(정제·파생 컬럼이 뒤 단계로 이어집니다).
2. 각 단계의 **답안 셀**(`# 여기에 코드를 작성하세요`)을 채웁니다. 정량 단계는 아래 **자가채점 셀**로 확인하고, **그래프 단계는 자가채점 없이** 위 **완성 그래프(정답)** 와 같은 모양으로 그립니다.
3. 마지막 **인사이트 단계**는 서술형입니다 — 정답 노트북의 모범 서술과 비교하세요.

화이팅!

In [ ]:
# [제공 코드] 통계 분석에 쓸 라이브러리와 한글 폰트를 준비합니다.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지
sns.set_theme(font=KOREAN_FONT, rc={"axes.unicode_minus": False})

## 데이터 살펴보기 — 먼저 데이터를 이해합니다
문제를 풀기 전에 **어떤 데이터인지 먼저 파악**합니다. `head()` 로 앞부분을, `info()` 로 열·자료형·결측을, `describe()` 로 수치·범주 요약을 봅니다. (아래 셀은 실행만 하면 됩니다.)

이 데이터는 한 유통사의 고객 2,240명 기록입니다. 소득(`Income`)에 결측이 있고, 결혼상태(`Marital_Status`)에 정상 범주가 아닌 오염값(`Absurd`·`YOLO`·`Alone`)이 섞여 있어 **정제가 필요한 실전 데이터**입니다.

In [ ]:
# [제공 코드] 데이터를 먼저 살펴봅니다 — 앞부분·구조·수치/범주 요약
#   (미리보기 전용 변수 preview 를 씁니다. 문제 풀이용 df 는 1단계에서 직접 불러오세요.)
preview = pd.read_csv("data/marketing_campaign.csv")
print("행·열 크기:", preview.shape)
print("\n[앞 5행] head()"); display(preview.head())
print("\n[열·자료형·결측] info()"); preview.info()
print("\n[수치 요약] describe()"); display(preview.describe())
print("\n[범주 요약] describe(exclude='number')"); display(preview.describe(exclude="number"))

## 1. 고객 기술통계 리포트
**배경**: 마케팅팀이 고객의 **소득과 지출**을 한 장으로 요약해 달라고 요청했습니다. 원본에는 결측과 오염값이 있으므로 먼저 **정제**한 뒤, 대표값·산포·분포·상관·신뢰구간까지 기술통계로 리포트를 완성합니다.

아래 각 `### N단계` 셀의 지시대로 **하나의 `df` 를 이어서** 분석합니다(2단계 정제·3단계 파생 컬럼이 뒤로 이어집니다).

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 원본 `(2240, 29)`, `Income` 결측 24개, `Marital_Status` 8범주(오염 포함) |
| 2단계 | 정제 후 행수 2233, `Income` 결측 0, 오염 결혼상태 0 |
| 3단계 | `total_spend` 평균 605.9, `age` 중앙값 44 |
| 4단계 | `Income` 평균 52234.71·중앙 51381.5·표준편차 25062.79·CV 47.98·이상치 8개 |
| 5단계 | 총지출 분포 히스토그램(왜도 표기) — 완성 그래프처럼 |
| 6단계 | 수치 4열 상관행렬 히트맵 — 완성 그래프처럼 |
| 7단계 | `Income` 평균 95% 신뢰구간 [51195.19, 53274.23] |
| 8단계 | 인사이트 서술(3문장 이상) |

### 1단계 — 데이터 불러오기·구조 파악
`data/marketing_campaign.csv` 를 `df` 로 불러오고, `df.shape`, `df["Income"].isna().sum()`(소득 결측 수), `df["Marital_Status"].value_counts()`(결혼상태 범주별 개수) 를 출력하세요.

- **요구사항**: 원본은 `(2240, 29)` 이고, `Income` 결측은 **24개**, `Marital_Status` 는 정상 6범주에 오염값 `Absurd`·`YOLO`·`Alone` 이 섞여 **총 8범주**입니다.
- **주의**: 아직 정제하지 않은 **원본 그대로**의 값을 확인하는 단계입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 파일을 데이터프레임으로 읽고 크기·결측 수·범주별 개수를 각각 출력한다.

세부구현:
1. read_csv 로 데이터를 df 에 담는다
2. shape 로 행·열 크기를 출력한다
3. Income 열의 결측 개수(isna 합)를 출력한다
4. Marital_Status 의 범주별 개수를 출력한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape == (2240, 29)
assert int(df["Income"].isna().sum()) == 24
assert df["Marital_Status"].nunique() == 8
print("✅ 1단계 통과!")

### 2단계 — 정제 (결측 대체 · 오염값 제거)
같은 `df` 를 다음 **순서**로 정제하세요.

1. **소득 결측 대체**: `Income` 의 결측을 `Income` 열의 **중앙값**(`df["Income"].median()`)으로 채웁니다.
2. **오염값 제거**: `Marital_Status` 가 `Absurd`·`YOLO`·`Alone` 인 행을 **삭제**합니다.

- **요구사항**: 정제 후 `df` 의 **행수는 2233**, `Income` 결측은 **0**, 오염 결혼상태는 **0개**여야 합니다.
- **주의**: 결측을 먼저 채운 **뒤** 행을 삭제하세요(순서가 중앙값에 영향). 삭제 후 `df` 를 그대로 이어 씁니다.
- **처음이라도 괜찮아요**: 여기 쓰는 `fillna`·`median`·`isin` 은 앞선 pandas 단원에서 익힌 기능입니다. 낯설면 아래 힌트를 펼쳐 네 단계를 그대로 따라오세요.

<details><summary>힌트</summary>

```text
접근방법:
- 정제는 '결측 채우기 → 오염 행 버리기' 두 단계다. 순서를 지켜 결측을 먼저 메운 뒤 행을 삭제한다.
- fillna 는 결측(NaN)을 괄호 안에 넣은 값으로 바꿔 준다. median 으로 그 열의 중앙값을 먼저 구해 fillna 에 넘긴다.
- isin 은 값이 목록 안에 있으면 참을 돌려준다. 그 앞에 물결표(~)를 붙이면 '목록에 없는' 행만 남길 수 있다.

세부구현:
1. Income 열의 중앙값을 median 으로 구한다
2. 그 중앙값을 fillna 에 넘겨 Income 의 결측을 채우고 결과를 다시 Income 열에 넣는다
3. 삭제할 오염값(Absurd·YOLO·Alone)을 리스트로 모은다
4. Marital_Status 가 그 목록에 드는지 isin 으로 표시하고, 앞에 물결표(~)를 붙여 목록에 없는 행만 골라 df 에 다시 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert df.shape[0] == 2233
assert int(df["Income"].isna().sum()) == 0
assert df["Marital_Status"].isin(["Absurd", "YOLO", "Alone"]).sum() == 0
print("✅ 2단계 통과!")

### 3단계 — 파생 변수 만들기 (총지출 · 나이)
정제된 `df` 에 두 파생 컬럼을 추가하세요.

- `total_spend`: 6개 지출 열(['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']) 의 **행별 합** — 고객 한 명의 총지출.
- `age`: **기준연도 2014** 에서 출생연도를 뺀 값 → `2014 - df["Year_Birth"]`.

- **요구사항**: `total_spend` 의 평균은 약 **605.9**, `age` 의 중앙값은 **44** 입니다.
- **주의**: 6개 지출 열의 합은 `df[열목록].sum(axis=1)` 처럼 **행 방향(axis=1)** 으로 더합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 새 컬럼은 df 의 새 이름 자리에 계산 결과를 대입해 추가한다.
- 여러 열을 '한 사람(행)별로' 더하려면 sum 에 axis 를 1(행 방향)로 준다. axis 를 빼면 열마다 세로로 더해 버리니 주의.

세부구현:
1. 지출 6열의 이름을 리스트로 모은다
2. 그 열들을 골라 sum 에 axis 1 을 주어 행마다 더해 total_spend 컬럼에 넣는다
3. 2014 에서 Year_Birth 를 빼 age 컬럼에 넣는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]  통계값은 정확일치 대신 허용오차(abs<tol)로 비교합니다.
assert "total_spend" in df.columns and "age" in df.columns
assert abs(float(df["total_spend"].mean()) - 605.9) < 0.01
assert int(df["age"].median()) == 44
print("✅ 3단계 통과!")

### 4단계 — 대표값·산포 (소득) 
`Income` 의 대표값과 산포를 구해 아래 이름의 변수에 담으세요.

- `inc_mean` = 평균(`mean`), `inc_median` = 중앙값(`median`), `inc_std` = 표준편차(**표본, `std(ddof=1)`**)
- `inc_cv` = **변동계수**(%) = `inc_std / inc_mean * 100`
- `inc_outliers` = **1.5×IQR 규칙 이상치 개수**(정수) — Q1·Q3 은 `quantile(0.25)`·`quantile(0.75)`, IQR = Q3−Q1, 경계 밖(`< Q1-1.5*IQR` 또는 `> Q3+1.5*IQR`) 개수.

- **요구사항(반올림 자리)**: `inc_mean`→2자리 52234.71, `inc_median`→2자리 51381.5, `inc_std`→2자리 25062.79, `inc_cv`→2자리 47.98, `inc_outliers`→정수 8.
- **주의**: 평균이 중앙값보다 큰 것은 **오른쪽 꼬리(고소득 이상치)** 때문입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 평균·중앙·표준편차를 구하고, 변동계수는 표준편차를 평균으로 나눠 100 을 곱한다.
- 이상치는 사분위와 IQR 로 위아래 경계를 만들어 그 밖의 개수를 센다.

세부구현:
1. Income 의 평균·중앙값·표준편차(ddof=1)를 각각 변수에 담는다
2. 변동계수 = 표준편차 / 평균 * 100
3. 1사분위·3사분위를 구해 IQR 과 아래·위 경계를 만든다
4. 경계 밖에 있는 값의 개수를 세어 정수로 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(inc_mean) - 52234.71) < 0.01
assert abs(float(inc_median) - 51381.5) < 0.01
assert abs(float(inc_std) - 25062.79) < 0.01
assert abs(float(inc_cv) - 47.98) < 0.01
assert inc_outliers == 8
print("✅ 4단계 통과!")

### 5단계 — 총지출 분포 (히스토그램 + 왜도)
`total_spend` 의 분포를 **히스토그램**으로 그리고, 분포의 비대칭을 나타내는 **왜도(skew)** 값을 제목에 표기하세요.

- `skew_val` = `stats.skew(df["total_spend"])` (약 0.86 — 오른쪽으로 꼬리가 긴 분포).
- `sns.histplot(data=df, x="total_spend", bins=40)` 로 그리고, 제목에 왜도 값을 넣습니다(예: `총지출 분포 — 왜도 0.86`).
- 그래프가 겹치지 않도록 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 와 같은 모양으로 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q1_s5.png" width="560"/>

In [ ]:
# 여기에 코드를 작성하세요

### 6단계 — 상관행렬 히트맵 (수치 4열)
소득·지출·구매횟수가 서로 어떻게 움직이는지 **상관행렬 히트맵**으로 보세요.

- 대상 4열: `["Income", "total_spend", "NumWebPurchases", "NumStorePurchases"]`
- `corr = df[대상4열].corr()` 로 상관행렬을 만들고, `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)` 로 그립니다.
- 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계도 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 처럼 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q1_s6.png" width="520"/>

In [ ]:
# 여기에 코드를 작성하세요

### 7단계 — 소득 평균의 95% 신뢰구간
`Income` **평균**의 **95% 신뢰구간**을 정규근사 공식으로 구해 `ci_low`·`ci_high` 에 담으세요.

- 신뢰구간은 **표본평균을 중심으로 표준오차만큼 좌우로 벌린 범위**입니다. 표본이 크므로 **정규근사**를 씁니다 — `scipy.stats` 에서 표준오차를 구하는 함수와, 신뢰수준·평균·표준오차를 받아 (하한, 상한)을 돌려주는 정규분포 구간 함수를 찾아 쓰세요.
- **주의**: `scale` 에 표준편차가 아니라 **표준오차**를 넣어야 평균의 신뢰구간이 됩니다.
- **요구사항(2자리 반올림)**: `ci_low` = **51195.19**, `ci_high` = **53274.23**.
- **주의**: 여기서는 신뢰구간 공식만 사용합니다(표본이 크므로 정규근사).

<details><summary>힌트</summary>

```text
접근방법:
- 평균과 표준오차를 구한 뒤, 정규분포 구간 함수에 신뢰수준·평균·표준오차를 넣는다.

세부구현:
1. Income 의 평균을 구한다
2. 표준오차를 sem 으로 구한다
3. norm.interval 에 0.95 와 평균·표준오차를 넣어 하한·상한을 받는다
4. 하한을 ci_low, 상한을 ci_high 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(ci_low) - 51195.19) < 0.01
assert abs(float(ci_high) - 53274.23) < 0.01
print("✅ 7단계 통과!")

### 8단계 — 인사이트 (서술)
위 대표값·산포·분포·상관·신뢰구간을 근거로 **고객의 소득과 지출 특징**을 **3문장 이상** 서술하세요.
- 평균과 중앙값의 관계(왜도), 이상치의 의미, 소득과 지출·구매의 상관을 말로 풀어 보세요.

*(여기에 3문장 이상으로 서술하세요 — 평균과 중앙값의 관계(왜도), 소득 이상치의 의미, 소득과 지출·구매의 상관, 신뢰구간이 말해 주는 것)*

## 2. 지출·반응 심화 리포트
**배경**: 이번에는 **어떤 고객이 더 많이 쓰고, 캠페인에 반응하는지** 를 집단별로 비교합니다. 교육수준·결혼상태별 지출 차이를 보고, 마지막 캠페인 반응(`Response`) 여부에 따른 지출 차이와, 총지출과 캠페인 수락의 관계를 기술통계로 정리합니다.

**최종 목표(자가채점 기준)**
| 단계 | 확인 항목 |
| --- | --- |
| 1단계 | 정제·파생 재현: 행수 2233, `total_spend` 평균 605.9 |
| 2단계 | 교육수준별 평균 — PhD 총지출 674.73·소득 56169.94, Basic 총지출 81.8 |
| 3단계 | 결혼상태별 총지출 박스플롯 — 완성 그래프처럼 |
| 4단계 | 반응별 총지출 평균 — 미반응 538.85·반응 991.24, 차이 452.39 |
| 5단계 | 총지출 ↔ 캠페인 수락 합 상관 r 0.46 |
| 6단계 | 인사이트 서술 |

### 1단계 — 불러오기·정제·파생 (문제 1 방식 재사용)
문제 2 를 위해 데이터를 **처음부터 다시** 불러와 문제 1 과 **같은 방식**으로 정제·파생합니다.

1. `data/marketing_campaign.csv` 를 `df` 로 불러온다.
2. `Income` 결측을 중앙값으로 채운다.
3. `Marital_Status` 가 `Absurd`·`YOLO`·`Alone` 인 행을 제거한다.
4. `total_spend` = 6개 지출 열(['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']) 의 행별 합 을 만든다.

- **요구사항**: 정제 후 행수 **2233**, `total_spend` 평균 약 **605.9**.
- **주의**: 문제 1 에서 만든 `df` 에 이어 쓰지 말고 **새로 로드**하세요(문제 간 오염 방지).

<details><summary>힌트</summary>

```text
접근방법:
- 문제 1 의 2·3단계에서 한 정제·파생을 그대로 다시 한다: fillna 로 결측 채우기 → isin 을 물결표(~)로 부정해 오염 행 버리기 → 지출 6열 합.

세부구현:
1. read_csv 로 원본을 새로 읽어 df 에 담는다
2. Income 의 중앙값(median)을 fillna 에 넘겨 결측을 채운다
3. Marital_Status 가 오염 목록에 없는 행만 isin 과 물결표(~)로 남긴다
4. 지출 6열을 골라 sum 에 axis 1 을 주어 행마다 더해 total_spend 를 만든다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert "age" not in df.columns, "문제 1 의 df 를 이어 쓰지 말고 원본을 새로 불러오세요"
assert df.shape[0] == 2233
assert abs(float(df["total_spend"].mean()) - 605.9) < 0.01
print("✅ 1단계 통과!")

### 2단계 — 교육수준별 소득·지출 비교 (groupby)
교육수준(`Education`)별로 **평균 소득과 평균 총지출**을 집계해 `edu_stats` 에 담으세요.

- `Education` 으로 묶어 `Income`·`total_spend` 두 열의 **평균**을 구해 `edu_stats` 에 담으세요 — 행은 교육수준(5범주), 열은 그 두 개라 모양은 `(5, 2)` 입니다.

- **요구사항(2자리 반올림)**: `edu_stats.loc["PhD", "total_spend"]` = **674.73**, `edu_stats.loc["PhD", "Income"]` = **56169.94**, `edu_stats.loc["Basic", "total_spend"]` = **81.8**.
- **주의**: 교육수준은 5범주(`Basic`·`2n Cycle`·`Graduation`·`Master`·`PhD`) 이므로 `edu_stats` 모양은 `(5, 2)` 입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 집단별 요약은 groupby 다. '무엇으로 나눌지'(Education)를 groupby 에 주고, '무슨 열'(Income·total_spend)의 '무슨 통계'(mean)를 볼지 정한다.
- groupby 에 기준 열을 준 뒤 대괄호로 볼 열 두 개를 고르고 mean 을 붙이면 '집단×통계' 표가 나온다. 위 요구사항에 그 형태가 그대로 적혀 있다.

세부구현:
1. Education 으로 groupby 한다
2. 대괄호로 Income·total_spend 두 열만 고른다
3. mean 으로 평균을 구해 edu_stats 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert edu_stats.shape == (5, 2)
assert abs(float(edu_stats.loc["PhD", "total_spend"]) - 674.73) < 0.01
assert abs(float(edu_stats.loc["PhD", "Income"]) - 56169.94) < 0.01
assert abs(float(edu_stats.loc["Basic", "total_spend"]) - 81.8) < 0.01
print("✅ 2단계 통과!")

### 3단계 — 결혼상태별 총지출 분포 (박스플롯)
결혼상태(`Marital_Status`)별 `total_spend` 분포를 **박스플롯**으로 비교하세요.

- `sns.boxplot(data=df, x="Marital_Status", y="total_spend")` 로 그리고 제목·축 이름을 답니다.
- 그리기 직전에 `plt.figure()` 를 호출하세요.

이 단계는 **자가채점이 없습니다** — 아래 **완성 그래프(정답)** 처럼 그리면 됩니다.

> **완성 그래프(정답)** — 아래 그림과 같은 모양이 나오도록 그려 보세요.

<img src="images/과제/lv3_q2_s3.png" width="620"/>

In [ ]:
# 여기에 코드를 작성하세요

### 4단계 — 캠페인 반응별 총지출 평균 차이 (기술통계)
마지막 캠페인 반응 여부(`Response`, 0=미반응·1=반응)로 나눠 **총지출 평균**을 비교하세요.

- `resp0_mean` = 미반응(0) 그룹의 `total_spend` 평균, `resp1_mean` = 반응(1) 그룹의 평균, `spend_diff` = `resp1_mean - resp0_mean` (반응 그룹이 얼마나 더 쓰는지).

- **요구사항(2자리 반올림)**: `resp0_mean` = **538.85**, `resp1_mean` = **991.24**, `spend_diff` = **452.39**.
- **주의**: 이 단원은 기술통계만 씁니다 — 여기서는 두 집단 평균의 차이만 구합니다(더 깊은 비교는 다음 단원).

<details><summary>힌트</summary>

```text
접근방법:
- 반응 여부로 두 그룹을 나눠 각각 총지출 평균을 구하고 그 차이를 계산한다.

세부구현:
1. Response 가 0 인 행의 total_spend 평균을 구한다
2. Response 가 1 인 행의 total_spend 평균을 구한다
3. 반응 평균에서 미반응 평균을 빼 차이를 구한다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(resp0_mean) - 538.85) < 0.01
assert abs(float(resp1_mean) - 991.24) < 0.01
assert abs(float(spend_diff) - 452.39) < 0.01
print("✅ 4단계 통과!")

### 5단계 — 총지출과 캠페인 수락의 상관
고객이 **여러 캠페인을 수락할수록 더 많이 쓰는지** 를 상관계수로 확인하세요.

1. `accepted_total` = 5개 캠페인 수락 열(['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']) 의 **행별 합**(0~5) 을 만듭니다.
2. `r_spend_cmp` = `total_spend` 와 `accepted_total` 의 **피어슨 상관계수** — `stats.pearsonr(...)` 의 첫 번째 반환값.

- **요구사항(2자리 반올림)**: `r_spend_cmp` = **0.46** (양의 상관 — 수락 캠페인이 많을수록 지출이 큰 편).
- **주의**: `pearsonr` 은 **두 값**을 돌려줍니다 — **첫 번째(상관계수)** 만 씁니다.

<details><summary>힌트</summary>

```text
접근방법:
- 캠페인 수락 5열을 행 방향으로 더해 수락 개수를 만들고, 총지출과의 피어슨 상관을 구한다.

세부구현:
1. 캠페인 수락 5열을 axis=1 로 합해 accepted_total 을 만든다
2. pearsonr 에 total_spend 와 accepted_total 을 넣는다
3. 반환값의 첫 번째(상관계수)를 r_spend_cmp 에 담는다
```

</details>

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
assert abs(float(r_spend_cmp) - 0.46) < 0.01
print("✅ 5단계 통과!")

### 6단계 — 인사이트 (서술)
위 집단별 비교와 상관을 근거로 **어떤 고객이 더 많이 쓰고 캠페인에 반응하는지** 를 **3문장 이상** 서술하세요.
- 교육수준·결혼상태별 지출 차이, 반응 그룹의 지출 차이, 수락 캠페인 수와 지출의 관계를 엮어 보세요.

*(여기에 3문장 이상으로 서술하세요 — 교육수준별 소득·지출 차이, 결혼상태별 지출 분포의 특징, 캠페인에 반응한 고객의 지출 수준, 총지출과 수락 수의 상관)*